# Creating "Principle Components Analysis (PCA) of field" from the Meteorological Predictor Fields as Input for RF-based ML-Models
Version 15 December 2022, Selina Kiefer

### Input: csv-files
continuous timeseries of meteorological predictors as 2d-fields in csv-format
### Output: csv-file
continuous timeseries of the first 10 principle components of the meteorological predictors per date in csv-format

#### Define the paths' and files' names

In [1]:
# Set the needed path and file names.
PATH_defined_functions = './Defined_Functions/'

PATH_input_data = './Data_in_csv_Format/'
ifiles_input_data = ['era5_u10_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z100_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z250_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_z500_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_z850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_t850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_H850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_u300_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                    'era5_msl_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv']

PATH_output_file = './Predictors_PCA_of_Field/'
file_name_output_file = 'era5_pca_n10_u10_z100_z250_z500_z850_t850_u300_msl_60W_60E_20N_80N_1950_2020_lead_time_14d.csv'


#### Import the necessary packages and functions
Nothing needs to be changed here.

In [2]:
# Import the necessary python packages.
import yaml
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA 

In [3]:
# Import the necessary functions.
import sys
sys.path.insert(1,PATH_defined_functions)
from read_in_csv_data import *

#### List the predictors to be combined

In [4]:
# List the desired predictors. From all dataframes, only 1 predictor is taken (if more are 
# needed, list these input files multiple times in "ifiles_input_data"). The month should not be
# included in the predictors list since a PCA over the month is not sensible. Therefore, the 
# month is added at a later stage to the dataframe.
desired_predictors = ['u10', 'z100', 'z250', 'z500', 'z850', 't850', 'H850', 'u300', 'msl']
time_column_name = 'time'
number_of_latitudes = 40
number_of_longitudes = 81

#### Decide how many principle components should be calculated

In [5]:
# Set how many components should be used by the Principal Components Analysis (PCA). 
number_of_principle_components = 10
pca = PCA(n_components=number_of_principle_components)

#### Perform the PCA for every predictor field separately and then combine the principle components in one dataframe
Nothing needs to be changed here. 

In [6]:
# The PCA is performed for every day and every predictor field separately. 
# Therefore, one dataframe is read in and grouped by year, month and days (.groupby()). Then, 
# one day is selected (.iloc[]) and from the resulting dataframe only the predictor's column is
# taken and converted into a numpy array. Then, this column is reshaped into the dimensions of
# a field again (.reshape(latitude, longitude)). Then, the PCA is performed (pca.fit_transform).
# From the PCA, the singular values (= PCA loadings) are taken and written in to a list. This
# list is then appended to list containing all singular values of the predictor field for all
# days and from this list, all the singular values are written into a pandas dataframe.
field_one_variable_all_days = []
df_input_data_pca = pd.DataFrame()

for i in range(len(ifiles_input_data)):
    df_input_data_one_variable = read_in_csv_data(PATH_input_data, ifiles_input_data[i])
    
    df_input_data_one_variable[time_column_name] = pd.to_datetime(df_input_data_one_variable[time_column_name])
    df_input_data_one_variable = df_input_data_one_variable.set_index(time_column_name)
    ds_input_data_one_variable_grouped = df_input_data_one_variable.groupby([df_input_data_one_variable.index.year, df_input_data_one_variable.index.month, df_input_data_one_variable.index.day], as_index=False)
    
    df_input_data_one_variable_grouped = pd.DataFrame(ds_input_data_one_variable_grouped)

    for k in range(len(df_input_data_one_variable_grouped)):
        df_input_data_one_variable_one_day = df_input_data_one_variable_grouped.iloc[k]
        df_input_data_one_variable_one_day = df_input_data_one_variable_one_day[1]

        field_one_variable_one_day = np.array(df_input_data_one_variable_one_day[desired_predictors[i]])
        field_one_variable_one_day = field_one_variable_one_day.reshape(number_of_latitudes,number_of_longitudes)
        
        field_one_variable_all_days.append(field_one_variable_one_day)
        
    field_one_variable_all_days = np.array(field_one_variable_all_days)
    
    field_one_variable_all_days = field_one_variable_all_days.reshape(( -1, number_of_latitudes*number_of_longitudes))
    
    field_one_variable_all_days_fitted = pca.fit_transform(field_one_variable_all_days)
              
    field_one_variable_all_days_transformed = pca.transform(field_one_variable_all_days)
   
    field_one_variable_all_days = []
            
    for l in range(number_of_principle_components):    
        df_input_data_pca[desired_predictors[i]+'_n'+str(l+1)] = field_one_variable_all_days_transformed[:,l]

#### Add the time information again to the reshaped data
Nothing needs to be changed here.

In [7]:
# Since the time got lost by using .groupby() and is not needed for the PCA, a separate new 
# dataframe is created containing only the time. To this dataframe, three new columns are added
# containing the year, month and day.
df_input_data_one_variable = df_input_data_one_variable.reset_index()
df_time = pd.DataFrame()
df_time[time_column_name] = pd.to_datetime(df_input_data_one_variable[time_column_name])
df_time = df_time.set_index(time_column_name)
df_time['year'] = df_time.index.year
df_time['month'] = df_time.index.month
df_time['day'] = df_time.index.day
df_time = df_time.reset_index()

In [8]:
# The separated timeseries is now combined into a single daily timeseries.
daily_timeseries = (df_time['year'].astype(str)+'-'+df_time['month'].astype(str)+'-'+df_time['day'].astype(str))

In [9]:
# In the next step, firstly the month is added to the dataframe containing the singular values
# of the PCA and then the time.
df_input_data_pca.insert(0, 'month', df_time['month'])
df_input_data_pca.insert(0, time_column_name, daily_timeseries)

#### Doublecheck the representation of the data

In [10]:
# Check if everything is reshaped and sorted correctly.
df_input_data_pca.head()

,time,month,u10_n1,u10_n2,u10_n3,u10_n4,u10_n5,u10_n6,u10_n7,u10_n8,...,msl_n1,msl_n2,msl_n3,msl_n4,msl_n5,msl_n6,msl_n7,msl_n8,msl_n9,msl_n10
0,1950-1-1,1,752.152426,566.393816,-17.889083,-38.884792,-97.191265,86.743304,386.914284,-71.828726,...,-32588.262593,-24714.141117,-4346.515047,-5446.885984,-23573.355600,-15096.470621,-30130.588627,4855.976217,803.887371,3839.808057
1,1950-1-2,1,683.078994,555.970008,-83.375763,15.672842,-92.121108,111.942614,354.373268,-88.633377,...,-26681.932350,-978.987608,-1905.988944,-9890.641274,-25315.398692,-9853.197543,-26218.342777,5821.875253,-3046.435104,-9341.044224
2,1950-1-3,1,661.344751,505.809017,-166.790105,67.314983,-56.519241,110.076034,348.076825,-75.326494,...,-30873.072165,5956.543070,3382.953271,-10713.431858,-34710.567958,-8229.787047,-8620.827757,2838.861891,-16014.899482,-6201.606194
3,1950-1-4,1,662.706781,459.729689,-275.907996,118.579985,15.681630,103.870503,348.917645,-66.475203,...,-29986.851584,-522.535333,10164.919825,-12724.645552,-36908.853039,-13503.117720,1823.874242,-249.549389,-12839.199224,3854.372159
4,1950-1-5,1,562.950666,521.793800,-377.547785,121.243776,124.502812,58.024040,355.133153,-53.445378,...,-13740.658079,-17752.886996,21938.504257,1267.639368,-35484.017179,3094.987403,1114.323381,2080.157119,2773.875893,10339.590815


In [11]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_input_data_pca.tail()

,time,month,u10_n1,u10_n2,u10_n3,u10_n4,u10_n5,u10_n6,u10_n7,u10_n8,...,msl_n1,msl_n2,msl_n3,msl_n4,msl_n5,msl_n6,msl_n7,msl_n8,msl_n9,msl_n10
12850,2020-12-13,12,-375.405467,-666.503271,-462.201468,-387.191991,216.137150,-128.379916,-80.070016,204.931585,...,-18776.886507,-24691.214793,36828.707821,-4480.218891,4991.923384,-14581.674188,17141.217563,12296.679735,-21294.498202,-4087.667984
12851,2020-12-14,12,-348.470262,-656.291429,-414.597932,-439.714031,314.229422,-98.156569,-152.102725,228.754905,...,-16945.611411,-18451.689360,42337.618364,-8447.138805,10311.856321,-15019.066145,5578.230946,7619.685663,-8936.126621,2352.193307
12852,2020-12-15,12,-377.169089,-605.245602,-336.715398,-533.562394,355.844164,-119.623941,-234.766611,214.591554,...,-18247.614403,-8641.070831,38082.824431,5684.996267,16841.136487,-4326.795827,-3792.609257,9117.538408,747.355861,2146.314363
12853,2020-12-16,12,-476.550253,-471.973106,-196.339085,-623.652278,430.109995,-160.814259,-238.885771,176.763554,...,-22028.231065,-7600.278963,28827.592393,25879.836096,8904.039134,4861.241475,-6522.414633,20826.921497,6229.513339,-401.904654
12854,2020-12-17,12,-595.672947,-356.774956,66.580199,-652.004619,556.037488,-215.760651,-221.024962,126.628539,...,-12622.602901,-17666.994824,26658.521369,18890.297999,1326.121760,-13113.427799,-12081.752524,16364.716096,-2251.035774,-4108.686158


#### Save the data in csv format
Nothing needs to be changed here.

In [12]:
# Save the pandas dataframe in csv-format.
df_input_data_pca.to_csv(PATH_output_file+file_name_output_file)

In [13]:
# End of Program